Towards Fault Tolerant Compiling
-----------
Recent experiments demonstrating that error-corrected logical qubits can already achieve higher fidelities than physical qubits have us excited about Fault-Tolerant (FT) quantum compilation [1, 2]. In this tutorial, we will highlight some crucial differences between the synthesis of NISQ and FT circuits and discuss how BQSKit can be used to target this new domain.

This tutorial requires the `bqskit-ft` extension package and can be install via pip:
```bash
pip install bqskit-ft
```

Contents
-----------
* [Fault-Tolerant Synthesis in BQSKit](#fault-tolerant-synthesis-in-bqskit)
* [Fault-Tolerant Gate Sets](#fault-tolerant-gate-sets)
* [Under the Hood](#under-the-hood)
    * [Single Qubit Gates](#single-qubit-gates)
    * [Search-based Synthesis](#search-based-synthesis)
    * [Synthesis by Diagonalization](#synthesis-by-diagonalization)
    * [Synthesis with Constrained Optimization](#synthesis-with-constrained-optimization)
    * [Decomposing Parameterized Gates](#decomposing-parameterized-gates)

Fault Tolerant Synthesis in BQSKit
----------
Let's start with an example of a circuit that we'd like to transpile to the fault-tolerant Clifford+$T$ gate set. Converting to Clifford+$T$ requires that a `FaultTolerantModel` be specified as the `MachineModel`. As we're targeting the Clifford+$T$ gate set, we will use a `CliffordTModel`. Then, we can just use `bqskit.compile`.

In [1]:
from bqskit.ir import Circuit
from bqskit.ir.gates import CNOTGate
from bqskit.ir.gates import U3Gate

from numpy import pi

from utils.printing import print_circuit

# A toy circuit to transpile
example_circuit = Circuit(2)
example_circuit.append_gate(U3Gate(), [0], [pi/2, 0, pi])
example_circuit.append_gate(CNOTGate(), (0, 1))
example_circuit.append_gate(U3Gate(), [1], [0, pi/4, 0])

print('The target transpilation circuit contains:')
print_circuit(example_circuit)

ModuleNotFoundError: No module named 'utils'

In [2]:
from bqskit.compiler.compile import compile

from bqskit.ft.cliffordt.cliffordtmodel import CliffordTModel

# Set up the machine model and compile the circuit
machine = CliffordTModel(num_qudits=example_circuit.num_qudits)
result = compile(example_circuit, model=machine)

# Print out the resulting circuit
print_circuit(result)

HGate@(0,)
CNOTGate@(0, 1)
TGate@(1,)


Fault-Tolerant Gate Sets
----------
Compilers are most concerned about transforming circuits when targeting near-term devices so that successful execution is maximized while conforming to the underlying machine's physical constraints. Typically, this means that circuits are optimized to use as few multi-qudit gates as possible, while requiring that circuits be transpiled to a particular gate set and mapped to a restrictive topology. BQSKit provides techniques for addressing each of these aspects of the compilation pipeline.

Fault-tolerant (FT) computation poses different challenges. Unlike in the NISQ case, FT compilation assumes that robust error-corrected qubits store information. This feat is accomplished by adding a new layer to the quantum computing stack: a quantum error-correcting code [3, 4, 5]. This new protection does not come for free. Implementing arbitrary unitary operations on logical qubits becomes much more difficult for codes of interest. This is because these codes admit restrictive discrete gate sets, whereas physical qubits can implement continuous rotations. To make matters worse, only a subset of the logical gates needed for universal computation can be easily implemented. These gates are in the Clifford group (generated by $\{H, S, CNOT\}$), which contains some familiar faces such as the Pauli matrices ($\{X, Y, Z\}$).

To make our logical gate set universal, we must add a single non-Clifford gate to the mix [6]. The most popular non-Clifford gate considered is the $T$ gate, which we define as 
$$
T = \begin{bmatrix}
1 & 0 \\
0 & e^{i\frac{\pi}{4}}
\end{bmatrix}
$$
Even though we cannot implement the $T$ gate directly like we can Clifford gates, teleportation lets us inject special resource states into our circuit that can be used to implement non-Clifford gates. We can prepare these special "magic" states using a process called magic state distillation [7]. Because so much must happen behind the scenes to ensure that non-Clifford gates can be executed, they are typically $100$-$1000\times$ more expensive to implement than their Clifford counterparts. This marks a considerable departure from the optimization objective in NISQ-centric synthesis algorithms. Whereas before, the primary concern was minimizing multi-qubit gates (such as $CNOT$ s), now the goal is minimizing non-Clifford gates (such as $T$ gates).

Now we can consider the single-qubit Clifford+$T$ gate set, which is simply $\{H, T\}$ (note that $T^2=S$ and $T^4=Z$). Equipped with this gate set, the Solovay-Kitaev theorem tells us that we can implement an approximation of any single-qubit unitary up to a precision of $\epsilon$ in $O(log^{3.97}(\frac{1}{\epsilon}))$ gates. Along with the $CNOT$ gate, this allows us to approximate any $n$-qubit operation in a fault-tolerant way!

Under the Hood
----------
Above we saw that compiling to fault-tolerant gate sets is as simple as setting the `MachineModel` to a `FaultTolerantModel`. But BQSKit can perform more complex compilation tasks while targeting fault-tolerant gate sets. In the remaining sections, we will highlight several advanced techniques for fault-tolerant synthesis.

### Single Qubit Gates


In practice, the algorithm given by the Solovay-Kitaev theorem results in long sequences of gates when approximating single-qubit unitaries. The state of the art for ancilla-free approximate synthesis comes from the Ross-Selinger "gridsynth" algorithm [8]. This algorithm produces optimal length sequences of Clifford+$T$ gates approximating $R_Z(\theta)$ rotations. Any single-qubit unitary can be written as:
$$
    U = R_Z(\theta) \sqrt{X} R_Z(\phi) \sqrt{X} R_Z(\lambda)
$$
Therefore, gridsynth lets us implement arbitrary single-qubit rotations at most only $3\times$ away from optimal. Unless $R_Z(\theta)$ corresponds to gates already in our Clifford+$T$ gate set ($Z$, $S$, or $T$), each $R_Z(\theta)$ is decomposed into a sequence of around $25\log\frac{1}{\epsilon}$ gates (of which about $\frac{1}{3}$ are $T$ gates). We will make use of the fact that we can optimally compile $R_Z(\theta)$ rotations in our approach to implementing multi-qubit unitaries in FT gate sets.

For the sake of this tutorial, we will target a gate set of Clifford+$R_Z(\theta)$. Post-processing can optionally be used to convert continuous $R_Z$ gates to discrete gates (see [Decomposing Parameterized Gates](#decomposing-parameterized-gates)).

Synthesis algorithms targeting the Clifford+$T$ gate set suffer a new challenge: how do you implement complex unitaries in discrete gate sets? We will present two methods of addressing this issue: 1) search-based techniques and 2) constrained optimization.

### Search-Based Synthesis
Search-based synthesis algorithms search through the discrete space of candidate circuits. Although the search space grows as $O(|\text{gateset}|^{\text{depth}})$, this approach can still be rather effective in certain cases.

Say we want to synthesize the following unitary:

In [3]:
from numpy import array

easy_target = array([
    [0.70710678, 0, 0.70710678, 0],
    [0, 0.5+0.5j, 0, 0.5+0.5j],
    [0.5+0.5j, 0, -0.5-0.5j, 0],
    [0, 0.70710678, 0, -0.70710678],
])

We can perform A* search over discrete gate sets in BQSKit by using a `DiscreteLayerGenerator`. Furthermore, we want to make sure synthesis gives us a very good implementation. We're going to set the `success_threshold=1e-8` to ensure this. The following block should take at most a few minutes.

In [4]:
from utils.discrete import DiscreteLayerGenerator

from bqskit.passes.synthesis import QSearchSynthesisPass
from bqskit.compiler import Compiler
from bqskit.ir import Circuit

success_threshold = 1e-8
layer_generator = DiscreteLayerGenerator()
synthesis_pass = QSearchSynthesisPass(
    success_threshold=success_threshold,
    layer_generator=layer_generator,
)

circuit = Circuit.from_unitary(easy_target)

with Compiler() as compiler:
    circuit = compiler.compile(circuit, [synthesis_pass])

print_circuit(circuit)

HGate@(0,)
CNOTGate@(0, 1)
TGate@(1,)
CNOTGate@(0, 1)


In this case, we were able to find a solution! Now let's say we want to try a slightly different unitary:

In [5]:
from numpy import array

hard_target = array([
   [0.70370187-0.06930858j, 0, 0.70370187-0.06930858j, 0],
   [0, 0.70370187+0.06930858j, 0, 0.70370187+0.06930858j],
   [0.70370187+0.06930858j, 0, -0.70370187-0.06930858j, 0],
   [0, 0.70370187-0.06930858j, 0, -0.70370187+0.06930858j],
])

What happens if you try to use the `DiscreteLayerGenerator` to synthesize this unitary?

(Note: you may have to restart the kernel after this part)

In [ ]:
# Use the DiscreteLayerGenerator to try synthesizing hard_target
circuit = Circuit.from_unitary(hard_target)

with Compiler() as compiler:
    circuit = compiler.compile(circuit, [synthesis_pass])

print_circuit(circuit)

### Synthesis by Diagonalization
Pure search-based synthesis has its limits. If the unitary is not easily implementable in the discrete gate set provided to the `DiscreteLayerGenerator`, the synthesis algorithm may spend lots of time checking different candidates before eventually timing out. Many techniques can push the boundaries of what's possible with pure discrete search, but the sheer size of the solution space will always limit these techniques. Finding solutions to a precision of `success_threshold=1e-3` might be possible in certain cases, but beyond that, for high precision, a new technique is needed.

To approach this problem, we're going to make an assumption: finding a solution circuit requires $R_Z(\theta)$ gates. Making this assumption means that our synthesis algorithm is unlikely to find optimal solutions, but a good, suboptimal solution with $R_Z(\theta)$ gates is better than no solution!

How can we modify the search-based synthesis framework to handle unitaries whose implementation requires $R_Z(\theta)$ gates? Using `QSearch` then converting every `U3Gate` to a sequence of `RZGate`s and `SqrtXGate`s is one option, which will be the topic of the next section, but there's another way.

We're going to assume the circuit implementing our `hard_target` unitary is in the form:
$$
    C(\bar\theta) = L \times D_{\bar\theta} \times R
$$
where $D_{\bar\theta}$ is a diagonal unitary, and the $L$ and $R$ subcircuits can be found by our search-based synthesis algorithm. We relax the need for our search-based synthesis algorithm to find sequences of gates that implement our target to just finding gates that diagonalize our target.

In [6]:
from bqskit.compiler import Compiler
from bqskit.ir.circuit import Circuit
from bqskit.passes.synthesis import QSearchSynthesisPass

from utils.diagonal_synth import replace_pauliz
from utils.discrete import DiscreteLayerGenerator
from utils.printing import print_circuit

success_threshold = 1e-8
# double_headed=True means we will try to diagonalize the target
layer_generator = DiscreteLayerGenerator(double_headed=True)
synthesis_pass = QSearchSynthesisPass(
    success_threshold=success_threshold,
    layer_generator=layer_generator,
)

circuit = Circuit.from_unitary(hard_target)

# Diagonalize
with Compiler() as compiler:
    circuit = compiler.compile(circuit, [synthesis_pass])

# Decompose diagonal component
circuit = replace_pauliz(circuit)

print_circuit(circuit)

HGate@(0,)
CNOTGate@(0, 1)
RZGate@(1,) [0.19634950428964368]
CNOTGate@(0, 1)


__Question:__ What similarities do you see between the outputs of synthesis for `easy_target` and `hard_target`?

__Answer:__ The outputs are structurally the same, but `hard_target` has a $\sqrt{T}$ gate instead of a $T$ gate between the two $CNOT$s. The `easy_target` can be very easily implemented with the Clifford+$T$ gate set. The $\sqrt{T}$ gate requires $O(log(1/\epsilon))$ Clifford+$T$ gates to implement.

__Question:__ Assume that the best implementation of `hard_target` corresponds to the results from above, but with the $R_Z$ operation converted to ${H,T}$ using gridsynth. Approximately how many candidate circuits might need to be searched to find this circuit without diagonalization? Assume `success_threshold=1e-8`.

__Answer:__ The $RZ$ gate requires approximately $25\log(1/10^{-8}) = 200$ gates to implement. The other three gates brings to total to 203. Given a gate set of $\{ CNOT, H, T \}$, we would need to look at $O(3^203)$ different circuits.

It turns out that there are many unitaries that appear in real quantum algorithms which are amenable to synthesis by Clifford+T diagonalization. Check out [9] if you'd like to learn about how we trained Reinforcement Learning agents to do exactly this!

### Synthesis with Constrained Optimization


For NISQ style circuit synthesis, BQSKit uses numerical instantiation to solve for the circuit parameters that approximate a target unitary. Using constrained optimization, we can apply that same principle here as well.

Using `optimization_level>=2` or providing a unitary as the target in `bqskit.compile` when targeting a `FaultTolerantModel` will automatically select this work flow.

In [7]:
from bqskit.qis.unitary import UnitaryMatrix
from numpy import array

hard_target = UnitaryMatrix(array([
   [0.70370187-0.06930858j, 0, 0.70370187-0.06930858j, 0],
   [0, 0.70370187+0.06930858j, 0, 0.70370187+0.06930858j],
   [0.70370187+0.06930858j, 0, -0.70370187-0.06930858j, 0],
   [0, 0.70370187-0.06930858j, 0, -0.70370187+0.06930858j],
]))

In [8]:
from bqskit.compiler.compile import compile

from bqskit.ft.cliffordt.cliffordtmodel import CliffordTModel

# Set up the machine model and synthesize the target unitary
machine = CliffordTModel(num_qudits=hard_target.num_qudits)
result = compile(hard_target, model=machine, optimization_level=3)

# Print out the resulting circuit
print_circuit(result)

dist = result.get_unitary().get_distance_from(hard_target)

from bqskit.ir.gates.constant.identity import IdentityGate
ft_gateset = list(machine.gate_set) + [IdentityGate()]

assert dist < 5e-8
assert all(op.gate in ft_gateset for op in result)

SqrtXGate@(0,)
SqrtXGate@(1,)
SqrtXGate@(0,)
SdgGate@(1,)
SdgGate@(0,)
SqrtXGate@(1,)
CNOTGate@(0, 1)
ZGate@(0,)
ZGate@(1,)
SqrtXGate@(0,)
RZGate@(0,) [6.0868357775195765]
SqrtXGate@(0,)
SdgGate@(0,)
CNOTGate@(0, 1)
HGate@(0,)
HGate@(1,)


Using numerical instantiation can result in extra Clifford gates compared to pure search-based approaches. Luckily, optimization of Clifford circuits is a well studied field, and Clifford circuits are relatively cheap to implement. Numerical instantiation based flows may be the only option for finding circuit implementations of complex targets. 

For details as to how contrained numerical optimization plays into instantiation, see [10].

### Decomposing Parameterized Gates

In the previous example, our multi-qubit synthesis algorithm produced a circuit that consist of Clifford gates and parameterized $R_Z(\theta)$. Fully fault-tolerant machines however are not likely to support these parameterized gate sets directly. This means we need to take every $R_Z(\theta)$ gate, and decompose them into sequences of single-qubit Clifford+$T$ gates (we will consider the set $\{H, X, S, T\}$).

Using `bqskit-ft`, this is done automatically when we compile using a `CliffordTModel`.

Let's try synthesizing the same unitary from before with this model instead.

Note that this next cell might take around 2 minutes to run.

In [ ]:
from bqskit.compiler.compile import compile

from bqskit.ft.cliffordt.cliffordtmodel import CliffordTModel

# Set up the machine model and synthesize the target unitary
machine = CliffordTModel(num_qudits=hard_target.num_qudits)
result_t = compile(hard_target, model=machine, optimization_level=3)

# Print out the resulting circuit
print_circuit(result_t)

dist_tar = result_t.get_unitary().get_distance_from(hard_target)
dist_rz = result_t.get_unitary().get_distance_from(result.get_unitary())

from bqskit.ir.gates.constant.identity import IdentityGate
ft_gateset = list(machine.gate_set) + [IdentityGate()]

assert dist_tar < 5e-8, 'Not close to the target!'
assert dist_rz < 5e-8, 'Not close to the RZ result!'
assert all(op.gate in ft_gateset for op in result_t)

We've now shown how we can take circuits or unitaries expressed in any gate set, then transpile them to the fault tolerant Clifford+$T$ gate set with `bqskit-ft`!

Remember that the easiest way to do this is to specify the model type as a `CliffordTModel` when calling the `compile` function.

## References
-------------
[1] https://arxiv.org/abs/2312.03982, D. Bluvstein, et al., "Logical quantum processor based on reconfigurable atom arrays"
<br>[2] https://arxiv.org/abs/2404.02280, M. P. da Silva, et al., "Demonstration of logical qubits and repeated error correction with better-than-physical error rates"
<br>[3] https://journals.aps.org/pra/abstract/10.1103/PhysRevA.52.R2493, P. Shor, "Scheme for reducing decoherence in quantum computer memory"
<br>[4] https://arxiv.org/abs/quant-ph/9601029, A. Steane, "Multiple Particle Interference and Quantum Error Correction"
<br>[5] https://arxiv.org/abs/quant-ph/9811052, S. Bravyi, A. Kitaev, "Quantum codes on a lattice with boundary"
<br>[6] https://iopscience.iop.org/article/10.1070/RM1997v052n06ABEH002155/meta, A. Kitaev, "Quantum computations: algorithms and error correction"
<br>[7] https://arxiv.org/abs/quant-ph/0403025, S. Bravyi, A. Kitaev, "Universal Quantum Computation with ideal Clifford gates and noisy ancillas"
<br>[8] https://arxiv.org/abs/1403.2975, N. Ross, P. Selinger, "Optimal ancilla-free Clifford+T approximations of z-rotations"
<br>[9] https://arxiv.org/abs/2409.00433, M. Weiden, et al., "Learning to Synthesize Fault-Tolerant Quantum Circuits by Diagonalization"
<br>[10] https://arxiv.org/abs/2504.15990, M. Weiden, et al., "Making Neural Networks More Suitable for Approximate Clifford+T Circuit Synthesis"
<br>[11] https://marcdav.is/static/fileshare/Numerical_Synthesis_of_Arbitrary-Multi-Qubit_Unitaries_with_Low_T-Count.pdf, M Grau Davis, "Numerical Synthesis of Arbitrary Multi-Qubit Unitaries with low T-Count"